# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamyaMuthumariappan/FlyRankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring (chosen in `w02_ml_task_framing.ipynb`).
**This week's job:** move the same contract from the 30k-row starter CSV onto the real ~79M-row
warehouse release, and prove every claim with a query instead of stating it.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RamyaMuthumariappan/FlyRankAI.git"
REPO_DIR = "FlyRankAI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

# --- Hugging Face token: env var -> Colab Secret -> prompt (last resort) ---
# Never paste the token into a cell — this repo is public.
import getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month — iterate here, NEVER on the _sample (that's June, the sealed test month)

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_month": f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')",
}

# Row counts are near-free (Parquet metadata, not data) — confirms we're pointed at the right thing.
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:16} {n:>12,} rows")

Working dir: /content/FlyRankAI
Paste your Hugging Face READ token (hf_...): ··········
dim_clients               104 rows
dim_content           519,606 rows
fact_daily_month    9,841,378 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item (page), for one client, for the calendar month `2026-03`.**
This is the same grain as the starter CSV used in `w01`/`w02` (one row per page) — the difference
is that here I *build* that monthly row myself, by aggregating the warehouse's finer-grained
daily fact table (`report_date × client_hash_id × content_hash_id`) up to `client_hash_id ×
content_hash_id` within one month partition.

**Table(s):**
- `fact_content_daily_performance`, partition `month=2026-03` — the primary source, aggregated to
  monthly grain (I read only this one partition, never the full 17-month table, and never the
  `_sample` table — that's June 2026, the sealed test month).
- `dim_content` (joined on `content_hash_id`) — content metadata/context.
- `dim_clients` (joined on `client_hash_id`) — per-client history coverage (`gsc_data_start`,
  `ga4_data_start`), used to sanity-check the panel, not as a feature.

**Time window:** `2026-03` (a mid-panel month, per the card's own example) for everything in this
notebook. The final month, `2026-06`, is treated as a sealed test month and is never touched here.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily_month']}
    LIMIT 5
""").df()


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**What I'd predict / rank (carried over from `w02`):** the same composite `priority_tier` proxy —
built from a decline signal plus severity context (how visible the page still is, how stale it
is) — that feeds a continuous priority *score*, which ranks all pages and hands the top slice to
an editor as a worklist. Nothing about moving to the warehouse changes that target; it changes
*where the inputs come from*.

**One thing I deliberately exclude:** any month-over-month delta built from the same impressions
split that would define the label (e.g. `(imp_last_half − imp_first_half) / imp_first_half`
inside the decision month). That's the warehouse-scale version of the starter CSV's
`trend_pct` / `trend_direction` rule — the quantity a decline label is computed *from* is never
allowed back in as a feature. Section 3's "trap" demonstrates exactly why.

| Bucket | Fields (this month's slice) | Why |
|---|---|---|
| **Context** (join/group only, never a feature) | `client_hash_id`, `content_hash_id`, `report_date` | Pseudonymous ids and dates — identify a row, don't describe it |
| **Feature** (knowable before the decision point) | `gsc_avg_position` (month avg, excl. 0/no-data), `gsc_impressions` (month sum), `gsc_clicks` (month sum), `days_with_impressions` (built), `ga4_data_available` share (built) | All measured *during* March itself, available the moment an editor would rank pages at start of April |
| **Label / proxy** (never a feature) | `is_declining_toy` (built for section 3's demo only, from a within-month first-half vs second-half impressions split) | The thing being predicted, or the raw signal it's computed from |
| **Excluded** | any first-half/second-half impressions *delta or pct-change* | Same signal the label is built from — feeding it back in is leakage, not a feature |

**Missingness note (carried forward from the flyrank-data skill):** `gsc_avg_position = 0` means
"no position data," not rank zero, and `ga4_data_available` can be `NULL` as well as `FALSE` —
both get handled explicitly in the queries below, not silently averaged in.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature / label / context / excluded buckets stated above — verified in section 3.")

Feature / label / context / excluded buckets stated above — verified in section 3.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three small proofs, all on `month=2026-03`, in order:

1. **Grain** — after aggregating the daily fact up to `client_hash_id × content_hash_id` for the
   month, is that pair really unique? (`GROUP BY ... HAVING COUNT(*) > 1` should come back empty.)
2. **Row count + date span** — how many (client, content) pairs does March actually contain, and
   does `MIN/MAX(report_date)` really cover the whole month?
3. **Availability** — filtering `ga4_data_available IS TRUE` (never `= TRUE`, because the flag can
   also be `NULL` — see the flyrank-data skill), how many row-days survive?

Then: a five-feature frame from that same March slice, and the deliberate leakage trap.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Query 1: GRAIN ---
# Build the monthly aggregate first (this IS my unit of analysis), then probe its grain.
monthly = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*)                                                   AS days_in_month,
        MIN(report_date)                                           AS first_day,
        MAX(report_date)                                           AS last_day,
        SUM(gsc_impressions)                                       AS impressions_mar,
        SUM(gsc_clicks)                                            AS clicks_mar,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_mar,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)       AS days_with_impressions,
        AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_available_share
    FROM {TABLES['fact_daily_month']}
    GROUP BY client_hash_id, content_hash_id
""").df()

grain_probe = monthly.groupby(["client_hash_id", "content_hash_id"]).size()
dupes = grain_probe[grain_probe > 1]
print(f"(client, content) pairs in March: {len(monthly):,}")
print(f"Pairs appearing more than once after aggregation: {len(dupes)}  <- should be 0")
assert len(dupes) == 0, "Grain does not hold — one row is not one content-item-month."
print("Grain holds: one row really is one content item, for one client, for March.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(client, content) pairs in March: 331,437
Pairs appearing more than once after aggregation: 0  <- should be 0
Grain holds: one row really is one content item, for one client, for March.


In [5]:
# --- Query 2: ROW COUNT + DATE SPAN ---
span = con.sql(f"""
    SELECT
        COUNT(*)                    AS raw_row_count,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date)             AS min_date,
        MAX(report_date)             AS max_date
    FROM {TABLES['fact_daily_month']}
""").df()
print("Raw daily rows in the March partition, and its date span:")
span

Raw daily rows in the March partition, and its date span:


,raw_row_count,n_clients,n_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [6]:
# --- Query 3: AVAILABILITY (IS TRUE, not = TRUE — the flag can be NULL too) ---
availability = con.sql(f"""
    SELECT
        COUNT(*)                                              AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END) AS ga4_available_true,
        SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_available_false,
        SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END) AS ga4_available_null
    FROM {TABLES['fact_daily_month']}
""").df()
print("Availability breakdown for March (row-days, not content items):")
print(availability)
survive = availability["ga4_available_true"].iloc[0]
total = availability["total_rows"].iloc[0]
print(f"\n{survive:,} / {total:,} row-days survive an `IS TRUE` filter "
      f"({survive/total:.1%}) — the rest are FALSE or NULL and would silently look like "
      f"'zero engagement' under a careless `= TRUE` / `!= FALSE` filter.")

Availability breakdown for March (row-days, not content items):
   total_rows  ga4_available_true  ga4_available_false  ga4_available_null
0     9841378            413966.0            6408671.0           3018741.0

413,966.0 / 9,841,378 row-days survive an `IS TRUE` filter (4.2%) — the rest are FALSE or NULL and would silently look like 'zero engagement' under a careless `= TRUE` / `!= FALSE` filter.


In [7]:
# --- Five features, built from the same March monthly aggregate above ---
features = monthly[[
    "client_hash_id", "content_hash_id",
    "avg_position_mar", "impressions_mar", "clicks_mar",
    "days_with_impressions", "ga4_available_share",
]].copy()

print(f"Feature frame: {len(features):,} content items x {features.shape[1]} columns")
features.head()

# 1. avg_position_mar        — knowable at the decision moment because it's the average GSC
#                               position measured DURING March itself, closed out before an
#                               editor would rank pages at the start of April.
# 2. impressions_mar         — knowable because it's a sum of March's own daily search impressions,
#                               fully observed by month-end.
# 3. clicks_mar               — same: an observed March total, not a forward-looking outcome.
# 4. days_with_impressions    — knowable because it counts March days already logged with >=1
#                               impression; no future information required.
# 5. ga4_available_share      — knowable because it's a coverage/availability flag over March's
#                               own days, not a measurement of what happens after March.
print()
print("available-when notes printed above each feature.")

Feature frame: 331,437 content items x 7 columns

available-when notes printed above each feature.


In [8]:
# --- THE TRAP: add one label-derived column on purpose ---
# Toy proxy (monthly granularity stands in for the real last30/prev30 split used in w02):
# split March into its first half and second half, and call a page "declining" if the
# second half lost >=20% of the first half's impressions. This mirrors is_declining_label
# from the starter CSV (built from trend_pct), performed here on real warehouse data.

halves = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_h1,
        SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_h2
    FROM {TABLES['fact_daily_month']}
    GROUP BY client_hash_id, content_hash_id
""").df()

trap = features.merge(halves, on=["client_hash_id", "content_hash_id"], how="left")
trap = trap[trap["imp_h1"] > 0].copy()   # need a non-zero base to define a % change at all
trap["is_declining_toy"] = (trap["imp_h2"] < 0.8 * trap["imp_h1"]).astype(int)
print("is_declining_toy rate:", trap["is_declining_toy"].mean().round(3))

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

honest_cols = ["avg_position_mar", "impressions_mar", "clicks_mar", "days_with_impressions"]
X = trap[honest_cols].fillna(0)
y = trap["is_declining_toy"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_acc = accuracy_score(y_te, honest_model.predict(X_te))
print(f"\nHonest accuracy (no leak, {len(honest_cols)} legit features): {honest_acc:.3f}")

# Now add the ONE label-derived column on purpose: the same pct-change the label came from.
trap["imp_pct_change_LEAK"] = (trap["imp_h2"] - trap["imp_h1"]) / trap["imp_h1"]
leak_cols = honest_cols + ["imp_pct_change_LEAK"]
X_leak = trap[leak_cols].fillna(0)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
leak_model = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
leak_acc = accuracy_score(y_te2, leak_model.predict(X_te2))
print(f"Leaked accuracy (same features + imp_pct_change_LEAK): {leak_acc:.3f}  <- jumps toward 1.0")

print(f"\nDelta from adding one label-derived column: {leak_acc - honest_acc:+.3f}")
print("Deleting imp_pct_change_LEAK now and keeping the HONEST number:", round(honest_acc, 3))
# imp_pct_change_LEAK is dropped — it never appears in `features` or `trap` again below this cell.
del trap["imp_pct_change_LEAK"]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_declining_toy rate: 0.327

Honest accuracy (no leak, 4 legit features): 0.711
Leaked accuracy (same features + imp_pct_change_LEAK): 0.999  <- jumps toward 1.0

Delta from adding one label-derived column: +0.288
Deleting imp_pct_change_LEAK now and keeping the HONEST number: 0.711


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** this March slice cannot tell me whether a page's March position/impressions
are typical for *that client*, because history depth is wildly unbalanced across clients — some
clients' `gsc_data_start` falls well before 2026-03, others start mid-panel or later (per
`dim_clients`). A page with a low March position could be a real page-one-decay case, or it could
simply be a client that only started being tracked in February — March would be its *first* real
month of data, not a decline. Any ranking built from this slice alone, without checking
`dim_clients.gsc_data_start` per client first, risks confusing "new to the panel" with "declining."
The `ga4_data_available IS TRUE/FALSE/NULL` three-way split in section 3 is the same problem one
level down: rows can be silently excluded from a naive `= TRUE` filter and misread as "no
engagement" when they are really "not tracked yet."

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
coverage = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        SUM(CASE WHEN gsc_data_start >= DATE '{MONTH}-01' THEN 1 ELSE 0 END) AS clients_new_this_month,
        SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS clients_null_start
    FROM {TABLES['dim_clients']}
""").df()
coverage

,n_clients,clients_new_this_month,clients_null_start
0,104,15.0,37.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.